# Nhiệm vụ 3: Đánh giá Chất lượng Dữ liệu cho Bài toán Hồi quy (Regression Readiness Audit)

**Thành viên thực hiện**: TV3 (Exploratory Data Analysis - EDA)  
**Mã học phần**: IT4930 - Khoa học Dữ liệu & Trí tuệ Kinh doanh  
**Tập dữ liệu**: `data/processed/products_processed.jsonl` ($N = 10.316$ listings)  
**Tài liệu quy chuẩn**: [docs/tv3/tv3.md](file:///d:/NMKHDL/it4930-ecommerce-data-intelligence/docs/tv3/tv3.md) & [docs/feature_contracts_tv3_tv8.md](file:///d:/NMKHDL/it4930-ecommerce-data-intelligence/docs/feature_contracts_tv3_tv8.md)

---
### Mục tiêu nghiên cứu:
1. **Nhận diện Ngoại lai (Outliers) & Dữ liệu biên**: Rà soát 1.768 bản ghi không hợp lệ (`is_price_missing == True`, `price_vnd <= 0`), lập bảng phân vị biên P0.1 - P99.9 và phân loại các nhóm ngoại lai cực biên (< 500k và > 100M).
2. **Hiện tượng Đa cộng tuyến (Multicollinearity)**: Đo lường tương quan chéo giữa các biến độc lập số và tính toán chỉ số phóng đại phương sai (**Variance Inflation Factor - VIF**), đưa ra cảnh báo cho mô hình tuyến tính OLS/Ridge/Lasso.
3. **Phòng chống Rò rỉ Dữ liệu (Data Leakage Risks Audit)**: Thiết lập ma trận rà soát Blacklist vs Whitelist (loại bỏ triệt để `discount_rate`, `original_price_vnd`, `listing_record_id`, `crawl_time`).
4. **Tác động của Trạng thái Tồn kho (`availability_status`)**: Đo lường hiện tượng lệch giá (**Price Drift**) của các sản phẩm `out_of_stock` (hàng ngừng kinh doanh EOL mang giá niêm yết cũ).
5. **Tuyên bố Khoa học Bắt buộc**: Các kết quả là bằng chứng khám phá thực nghiệm (*exploratory associations*), không cấu thành suy diễn nhân quả (*causal inference*).

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# Thiết lập import module EDA nội bộ
eda_scripts_path = Path("../scripts").resolve()
if str(eda_scripts_path) not in sys.path:
    sys.path.append(str(eda_scripts_path))

from eda_stats import (
    load_processed_products,
    audit_price_outliers_and_boundaries,
    compute_multicollinearity_and_vif,
    audit_data_leakage_risks,
    analyze_availability_status_impact
)
from eda_viz import (
    set_plotting_theme,
    plot_price_outliers_and_boundaries,
    plot_multicollinearity_and_vif,
    plot_availability_price_comparison
)

set_plotting_theme()
print("[*] Hệ thống thư viện EDA Task 3 đã sẵn sàng!")

In [ ]:
# Tải dữ liệu chuẩn hóa
data_path = "../../data/processed/products_processed.jsonl"
df = load_processed_products(data_path)
print(f"[+] Tổng số bản ghi quan sát: {len(df):,}")

## 1. Nhận diện Outliers & Dữ liệu biên (Task 3.1)
Kiểm toán các bản ghi thiếu giá, giá âm, phân vị cực biên và các trường hợp ngoại lai cực đoan.

In [ ]:
invalid_summary, boundary_df, outlier_summary = audit_price_outliers_and_boundaries(df)

print("=== BẢNG 3.1A: KIỂM TOÁN CÁC BẢN GHI GIÁ THIẾU VÀ KHÔNG HỢP LỆ ===")
display(invalid_summary)

print("\n=== BẢNG 3.1B: BẢNG PHÂN VỊ CỰC BIÊN VÀ TUKEY FENCES CỦA GIÁ BÁN ===")
display(boundary_df)

print("\n=== BẢNG 3.1C: NHẬN DIỆN VÀ ĐÁNH GIÁ CÁC NHÓM NGOẠI LAI CỰC BIÊN ===")
display(outlier_summary)

In [ ]:
plot_price_outliers_and_boundaries(df, "../outputs/figures/fig_3_1_price_outliers_and_boundaries.png")
Image("../outputs/figures/fig_3_1_price_outliers_and_boundaries.png")

## 2. Đo lường Hiện tượng Đa cộng tuyến & Tính toán VIF (Task 3.2)
Phân tích tương quan chéo giữa các biến độc lập và tính hệ số phóng đại phương sai (VIF).

In [ ]:
lap_corr, ph_corr, vif_df = compute_multicollinearity_and_vif(df)

print("=== BẢNG 3.2A: MA TRẬN TƯƠNG QUAN CHÉO LAPTOP ===")
display(lap_corr)

print("\n=== BẢNG 3.2B: MA TRẬN TƯƠNG QUAN CHÉO SMARTPHONE ===")
display(ph_corr)

print("\n=== BẢNG 3.2C: CHỈ SỐ VIF (VARIANCE INFLATION FACTOR) ===")
display(vif_df)

In [ ]:
plot_multicollinearity_and_vif(df, "../outputs/figures/fig_3_2_multicollinearity_and_vif.png")
Image("../outputs/figures/fig_3_2_multicollinearity_and_vif.png")

## 3. Phòng chống Rò rỉ Dữ liệu (Data Leakage Risks Audit - Task 3.3)
Xác định danh sách Blacklist (loại bỏ hoàn toàn) và Whitelist (candidate features an toàn).

In [ ]:
leakage_df = audit_data_leakage_risks()
print("=== BẢNG 3.3: MA TRẬN KIỂM TOÁN RỦI RO RÒ RỈ DỮ LIỆU (TV5) ===")
display(leakage_df)

## 4. Tác động của Trạng thái Tồn kho (`availability_status`) & Price Drift (Task 3.4)
Kiểm định xem các sản phẩm hết hàng (`out_of_stock`) có mang giá niêm yết cũ/lạc hậu hay không.

In [ ]:
status_stats, status_test = analyze_availability_status_impact(df)

print("=== BẢNG 3.4A: THỐNG KÊ GIÁ THEO TRẠNG THÁI TỒN KHO ===")
display(status_stats)

print("\n=== BẢNG 3.4B: KIỂM ĐỊNH MANN-WHITNEY U TEST VỀ HIỆN TƯỢNG PRICE DRIFT ===")
display(status_test)

In [ ]:
plot_availability_price_comparison(df, "../outputs/figures/fig_3_3_availability_price_comparison.png")
Image("../outputs/figures/fig_3_3_availability_price_comparison.png")